# Churn Prediction — Feature Engineering and Preprocessing

## Objective 

The objective of this notebook is to address the data quality issues and feature engineering decisions identified in `01_eda.ipynb` and prepare the dataset for modeling. 

**Data quality fixes:**
- Replace 11 `NaN` values in `TotalCharges` with `0.0` — the 11 were confirmed as data entry errors (`' '` instead of `0.0`) for brand new customers with `tenure = 0` 
- Drop `customerID` — it carries no predictive information 

**Feature engineering decisions:** 
- Encode binary categorical features (`Yes`/`No` → 0/1) 
- log transform `TotalCharges`
- One-hot encode multi-category features (`Contract`, `InternetService`, `PaymentMethod`) 
- Handle `No internet service` third category in service columns 
- Address redundancy among service features (`OnlineSecurity`, `TechSupport`, `OnlineBackup`, `DeviceProtection`) and between `StreamingTV` and `StreamingMovies` 
- Address confounding between `Contract`, `PaymentMethod`, and `PaperlessBilling` 
- Decide on `tenure` vs `TotalCharges` retention given 0.83 correlation 

Most transformations are fixed mappings applied before the train/test split. Scaling is applied after the split — fitted on training data only and then applied to both train and test to prevent data leakage. 

## Outputs 
- Cleaned, transformed, and scaled dataset saved to `data/processed/` as `X_train.csv`, `X_test.csv`, `y_train.csv`, and `y_test.csv`, ready for modeling in `03_modeling_and_evaluation.ipynb`.

## 2.1 Setup & Load Dataset

Importing some of the same core libraries as `01_eda.ipynb`, with the addition of `os` for directory management, and sklearn modules for train/test splitting and scaling. Path constants and `RANDOM_STATE` are defined here to ensure reproducibility and consistent file references throughout the notebook. `os.makedirs()` is called to ensure the `data/processed/` directory exists before saving — preventing errors when running on a fresh clone of the repo.

In [1]:
import os
import warnings
warnings.filterwarnings('ignore') # suppress sklearn warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
RAW_DATA_PATH = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
X_TRAIN_PATH = '../data/processed/X_train.csv'
X_TEST_PATH = '../data/processed/X_test.csv'
Y_TRAIN_PATH = '../data/processed/y_train.csv'
Y_TEST_PATH = '../data/processed/y_test.csv'

os.makedirs('../data/processed/', exist_ok=True)

Loading the raw dataset fresh from `data/raw/` using `RAW_DATA_PATH` to ensure all transformations in this notebook start from the original, unmodified data.

In [23]:
df = pd.read_csv(RAW_DATA_PATH)
print(df.shape)
pd.set_option('display.max_columns', None)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2.2 Target Binarization 

Converting the target variable `Churn` from string values (`Yes`/`No`) to binary (1/0) using `.map()`. This is required since sklearn models expect numeric targets, not string labels.

In [24]:
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})
df['Churn'].head()

0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64

`.head()` confirms `Churn` has been successfully converted to binary — `Yes`->`1` and `No`->`0`. The column is now ready to be used as the target variable in modeling.

## 2.3 Dropping `customerID`

`customerID` carries no predictive information — it's a unique identifier for each customer and would only add noise to the model. Dropping it using `.drop()` with `axis=1` to remove the entire column.

In [25]:
df.drop('customerID', axis=1, inplace=True)
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [10]:
df.shape

(7043, 20)

`.head()` and `.shape` confirm `customerID` has been successfully dropped — the column no longer appears in the dataset and `df.shape` returns `(7043, 20)`, confirming 20 remaining columns (19 features + target `Churn`).

## 2.4 Handling Missing Values

To handle the 11 missing values in `TotalCharges`, I will replace them with `0.0` — the correct value based on domain knowledge. To do this fixed replacement, I will first convert the column to numeric using `pd.to_numeric()` with `errors='coerce'` to turn the `' '` space characters into `NaN`, then use `.fillna(0.0)` to replace them with the correct value. 

In [26]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors = 'coerce')
df['TotalCharges'].fillna(0.0, inplace=True)

0         29.85
1       1889.50
2        108.15
3       1840.75
4        151.65
         ...   
7038    1990.50
7039    7362.90
7040     346.45
7041     306.60
7042    6844.50
Name: TotalCharges, Length: 7043, dtype: float64

In [27]:
print(df['TotalCharges'].dtype)
df['TotalCharges'].isnull().sum()

float64


np.int64(11)

`df['TotalCharges'].dtype` confirms the column is now `float64` and `df['TotalCharges'].isnull().sum()` returns 0 — confirming all 11 missing values have been correctly replaced with `0.0`.

## 2.5 Encoding Categorical Features

### Handling Third Categoires

The service columns (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) each contain a third category, `No internet service`, which is identical across all six columns since it represents the same underlying customer segment (no internet subscription). Rather than one-hot encoding this third category redundantly across six columns, I created a single binary feature `has_internet` to capture the internet/no-internet distinction once. The `No internet service` values are then replaced with `No` in the service columns, making them clean Yes/No features. This avoids redundancy while preserving the information — a customer with `has_internet=0` and `No` across service columns is correctly represented as a non-internet customer who has no add-on services.

Similarly, `MultipleLines` contains a `No phone service` third category. Since `PhoneService` already captures the has/hasn't phone distinction as a binary Yes/No column, I replace `No phone service` with `No` in `MultipleLines` without needing a separate `has_phone` feature.

In [28]:
# create has_internet binary feature
df['has_internet'] = (df['InternetService'] != 'No').astype(int)

# replace third categories in service columns
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in service_cols:
    df[col] = df[col].replace('No internet service', 'No')

# replace third categories in MultipleLines
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

In [17]:
print(df['OnlineSecurity'].value_counts())
df['MultipleLines'].value_counts()

OnlineSecurity
No     5024
Yes    2019
Name: count, dtype: int64


MultipleLines
No     4072
Yes    2971
Name: count, dtype: int64

In [20]:
print(df['has_internet'].value_counts())
df['InternetService'].value_counts()

has_internet
1    5517
0    1526
Name: count, dtype: int64


InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

`.value_counts()` confirms the service columns (`OnlineSecurity`, `OnlineBackup`, etc.) and `MultipleLines` now contain only `Yes`/`No` — the third categories (`No internet service`, `No phone service`) have been successfully replaced. `has_internet` was verified by comparing its value counts against `InternetService` — `has_internet = 0` returns 1,526 rows, exactly matching the `InternetService = No` count, confirming the binary feature was created correctly.

### Binary Encoding

Before binary encoding I drop `gender` since `gender` showed a negligible churn rate difference (Female 26.9% vs Male 26.2%) and was identified as the weakest predictor in EDA. All other weak predictors (`PhoneService`, `MultipleLines`, `StreamingTV`, `StreamingMovies`) will be evaluated in modeling via feature importance before deciding to drop. `df.head()` confirms the drop was successful.

In [ ]:
df.drop('gender', axis=1, inplace=True)
df.head()

,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,has_internet
0,0,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,1
1,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0,1
2,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,1
3,0,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0,1
4,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,1


The following columns contain binary `Yes`/`No` values and will be encoded to 1/0 using `.map()` to prepare them for modeling: 
- **Demographic:** `Partner`, `Dependents` 
- **Phone & lines:** `PhoneService`, `MultipleLines` 
- **Services:** `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`
- **Billing:** `PaperlessBilling`

In [ ]:
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

In [31]:
df[binary_cols].nunique()

Partner             2
Dependents          2
PhoneService        2
MultipleLines       2
OnlineSecurity      2
OnlineBackup        2
DeviceProtection    2
TechSupport         2
StreamingTV         2
StreamingMovies     2
PaperlessBilling    2
dtype: int64

`.nunique()` confirms the categories were encoded correctly (`Yes` -> 1, `No` -> 0) — every column returns exactly 2 unique values with dtype int64, which confirms there are no str values or `NaN`s. 